In [3]:
import pandas as pd
import numpy as np

CAMINHO_CSV = "data/raw/sigps_dataset.csv"
df = pd.read_csv(CAMINHO_CSV)

df.head(), df.shape

(   checkin_id  paciente_id   especialidade     tipo_fila    data_hora_checkin  \
 0           1         2620      psicologia         geral  2025-12-25T11:54:48   
 1           2         1732    fisioterapia         geral  2025-12-20T02:55:40   
 2           3         1621  fonoaudiologia         geral  2025-12-24T13:59:28   
 3           4         2267      psicologia  especialista  2025-12-11T01:16:54   
 4           5         2585      psicologia         geral  2026-02-20T05:33:26   
 
    urgencia  historico_consultas  historico_faltas  minutos_na_fila  \
 0         1                   14                 1               85   
 1         2                   48                 6               18   
 2         1                   15                 0               35   
 3         2                   56                 3               34   
 4         2                   55                 5               40   
 
    score_prioridade  
 0                36  
 1                19  
 2 

In [4]:
from src.features import ORDEM_FEATURES

# X com a ordem oficial
X = df[ORDEM_FEATURES].to_numpy(dtype=np.float32)

# y: se existir score_prioridade, usa. Se não, gera por heurística (fallback)
if "score_prioridade" in df.columns:
    y = df["score_prioridade"].to_numpy(dtype=np.float32)
else:
    urg = X[:, 0]
    hist_cons = X[:, 1]
    hist_falt = X[:, 2]
    min_fila = X[:, 3]
    raw = urg * 18.0 + min_fila * 0.20 + hist_cons * 0.25 - hist_falt * 6.0
    y = np.clip(raw, 0, 100).astype(np.float32)

# validações
print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("y min/max:", float(y.min()), float(y.max()))

X.shape: (20000, 4)
y.shape: (20000,)
y min/max: 0.0 100.0


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

modelo.fit(X_treino, y_treino)
pred = modelo.predict(X_teste)

mae = float(mean_absolute_error(y_teste, pred))
r2 = float(r2_score(y_teste, pred))

print("MAE:", mae)
print("R²:", r2)

MAE: 3.5904898081864567
R²: 0.9492494355314888


In [6]:
import json
from pathlib import Path
import joblib

PASTA_ARTIFACTS = Path("artifacts")
PASTA_ARTIFACTS.mkdir(parents=True, exist_ok=True)

CAMINHO_MODELO = PASTA_ARTIFACTS / "model.pkl"
CAMINHO_METRICAS = PASTA_ARTIFACTS / "metrics.json"

joblib.dump(modelo, CAMINHO_MODELO)

payload = {
    "mae": mae,
    "r2": r2,
    "ordem_features": ORDEM_FEATURES,
    "n_treino": int(len(X_treino)),
    "n_teste": int(len(X_teste)),
    "modelo": type(modelo).__name__,
    "dataset_csv": CAMINHO_CSV,
}

CAMINHO_METRICAS.write_text(json.dumps(payload, indent=2), encoding="utf-8")

print("Modelo salvo em:", CAMINHO_MODELO)
print("Métricas salvas em:", CAMINHO_METRICAS)

Modelo salvo em: artifacts\model.pkl
Métricas salvas em: artifacts\metrics.json


In [7]:
# Ordem: urgencia, historico_consultas, historico_faltas, minutos_na_fila
casos = {
    "A": [1, 1, 0, 5],
    "B": [5, 1, 0, 5],
    "C": [5, 1, 0, 120],
}

for nome, x in casos.items():
    score = float(modelo.predict([x])[0])
    print(nome, x, "=>", round(score, 2))

A [1, 1, 0, 5] => 18.96
B [5, 1, 0, 5] => 88.76
C [5, 1, 0, 120] => 99.89


In [8]:
faltando = [c for c in ORDEM_FEATURES if c not in df.columns]
print("Colunas faltando:", faltando)
print("Colunas do CSV:", list(df.columns))

Colunas faltando: []
Colunas do CSV: ['checkin_id', 'paciente_id', 'especialidade', 'tipo_fila', 'data_hora_checkin', 'urgencia', 'historico_consultas', 'historico_faltas', 'minutos_na_fila', 'score_prioridade']


In [15]:
df.to_csv(r'data\raw\treino.csv', index=False)